# Fourier neural operator: learning the Burgers solution operator

*others · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/others/neural_operator)

Instead of solving one equation, a Fourier neural operator (FNO) learns the whole map $u_0(x) \mapsto u(x, T)$ of the viscous Burgers equation $u_t + u\,u_x = \nu\,u_{xx}$ from pairs produced by a spectral solver. Its key layer is a spectral convolution: Fourier transform, mix channels mode by mode with complex weights $R_{iok}$, keep only the lowest $m$ modes, transform back. Because the weights live on Fourier modes, the trained model also runs on a finer grid than it was trained on.

**Vector form**

$$G_\theta : u_0(\cdot) \mapsto u(\cdot, T),\quad u_t + u\,u_x = \nu\,u_{xx},\qquad v^{(l+1)} = \sigma\big(W v^{(l)} + \mathcal F^{-1}[R\cdot\mathcal F v^{(l)}]\big),\qquad \min_\theta \tfrac1B\textstyle\sum_b \frac{\lVert G_\theta(u_0^b) - u_T^b\rVert}{\lVert u_T^b\rVert}$$

**Index (tensor) form**

$$\hat v_{bik} = \textstyle\sum_x v_{bix}\,e^{-ikx},\qquad \hat w_{bok} = R_{iok}\,\hat v_{bik}\ \ (k < m),\qquad v^{(l+1)}_{box} = \sigma\big(W_{oi}\,v^{(l)}_{bix} + b_o + \mathcal F^{-1}_{k\to x}[\hat w_{bok}]\big)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $a_{bfx}$ | `stack([u0, x / 2π], dim=1)` | (B, 2, N) | input function: $u_0$ and the position at each of the $N$ grid points of sample $b$ |
| $v_{bcx} = P_{cf}\,a_{bfx} + p_c$ | `einsum('cf,bfx->bcx', P, a) + p` | (B, C, N) | lift: $f$ summed, 2 features become $C$ channels at every point |
| $\hat v_{bik}$ | `torch.fft.rfft(v)` | (B, C, N/2+1) | complex Fourier coefficients of every channel $i$, wavenumber $k$ |
| $R_{iok}$ | `w["R"][l]` | (C, C, m) | complex weights, one $C\times C$ matrix per kept mode $k < m$: they do not depend on $N$ |
| $R_{iok}\,\hat v_{bik}$ | `einsum('bik,iok->bok', v_hat[..., :m], R)` | (B, C, m) | repeated $i$ ⇒ summed; $k$ is NOT summed: each mode is mixed separately |
| $\hat w_{bok} = 0,\ k \ge m$ | `out_hat = zeros_like(v_hat)` | (B, C, N/2+1) | truncation: only the lowest $m$ Fourier modes pass the spectral convolution |
| $\mathcal F^{-1}_{k\to x}$ | `torch.fft.irfft(out_hat, n=N)` | (B, C, N) | back to any number $N$ of grid points: this is where the resolution is chosen |
| $W_{oi}\,v_{bix} + b_o$ | `einsum('oi,bix->box', W, v) + b` | (B, C, N) | local path, the same $C\times C$ matrix at every point; $\sigma$ = GELU |
| $u_{bx}$ | `project(v, Q1, q1, Q2, q2)` | (B, N) | pointwise two-layer projection from $C$ channels to the predicted $u(x, T)$ |
| $\lVert G_\theta(u_0) - u_T\rVert / \lVert u_T\rVert$ | `relative_l2(pred, true)` | () | relative $L^2$ error per sample, averaged over the batch: the loss minimised by Adam |
| $\partial_t\hat u_k = -\nu k^2\hat u_k - \tfrac{ik}{2}\,\widehat{u^2}_k$ | `solve_burgers(u0, nu, t_final, dx)` | (B, N) | the data: pseudo-spectral solver, integrating-factor RK4 with the 2/3 dealiasing rule, all samples at once |

The training data are pairs $(u_0, u(\cdot, T))$: random smooth initial conditions (sums of Fourier modes $k \le 5$ with amplitudes falling like $1/k$) and their solutions, computed on 256 points by a pseudo-spectral solver and subsampled to 64 points for training. The network never sees the equation, only solver data. The tensor lesson is the spectral convolution: after rfft each channel $i$ of each sample $b$ becomes Fourier coefficients $\hat v_{bik}$, and einsum('bik,iok->bok') sums over the input channel $i$ but keeps the wavenumber $k$ as a free index, so each mode gets its own $C\times C$ complex matrix $R_{\cdot\cdot k}$. That is a convolution with a global, learned kernel, computed as a product in Fourier space. Higher modes $k \ge m$ are cut off; the local path $W_{oi}v_{bix}$ and the nonlinearity $\sigma$ put fine detail back. Nothing in $R_{iok}$, $W_{oi}$, $P$ or $Q$ refers to the number of grid points, so the trained operator runs on any grid with more than $2m$ points: the bottom-right panel feeds the SAME weights 256 points (zero-shot super-resolution) for the test case with the steepest front. The 64-point predictions (dots) lie on the 256-point curve: this discretisation invariance is approximate, because $\sigma$ creates harmonics that alias differently on different grids, but the error on 256 points equals the 64-point error. What limits the accuracy is the model, not the grid: near-shocks need modes above $m$ and are the hardest cases. Compare the PINN in this topic: a PINN solves ONE problem by putting the equation in the loss, and must be retrained for every new initial condition; a neural operator learns a whole family of problems from solver data, after which each new prediction is a single forward pass. The console prints the time of that forward pass against the solver: tens of times faster here, but for 1D Burgers the solver is already cheap and the data generation and training cost far more. Neural operators pay off where one simulation takes hours (weather, turbulence, porous-media flow) and many queries are needed, and only inside the distribution they were trained on. Try fewer modes $m$, a smaller training set, or a larger $\nu$ (smoother solutions, easier to learn). Simplifications: one output time $T$, a fixed seed, the grid position as a plain input channel and no padding (the domain is periodic anyway).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/others/neural_operator/main.py
"""Main loop: random initial conditions → solver data → train the Fourier neural operator → test → plot."""

import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors

N_COARSE = 64  # training resolution
N_FINE = 256  # solver resolution; also the zero-shot super-resolution test grid
N_TEST = 200  # held-out initial conditions
N_IC_MODES = 5  # u_0 is a sum of Fourier modes k = 1 … 5
N_LAYERS = 4
BATCH = 50
SEED = 0


def make_data(n_train: int, nu: float, t_final: float, dev: torch.device) -> dict:
    """Solve Burgers for n_train + N_TEST random initial conditions on the fine grid, then subsample to N_COARSE.

    Coarse tensors are float32 (for the network); the fine test set is kept in float64 for the solver.
    """
    x_fine, dx_fine = tensors.grid(N_FINE, dev)  # (M,)
    coeffs = tensors.random_modes(n_train + N_TEST, N_IC_MODES, SEED, dev)  # (B, K, 2)
    u0_fine = tensors.initial_conditions(coeffs, x_fine)  # (B, M)
    uT_fine = solve.solve_burgers(u0_fine, nu, t_final, dx_fine)  # (B, M)
    s = N_FINE // N_COARSE  # the coarse grid is every 4th fine point
    return {
        "x": x_fine[::s].float(),  # (N,)
        "u0_train": u0_fine[:n_train, ::s].float(),  # (n_train, N)
        "uT_train": uT_fine[:n_train, ::s].float(),
        "u0_test": u0_fine[n_train:, ::s].float(),  # (N_TEST, N)
        "uT_test": uT_fine[n_train:, ::s].float(),
        "x_fine": x_fine,  # (M,) float64
        "dx_fine": dx_fine,
        "u0_test_fine": u0_fine[n_train:],  # (N_TEST, M) float64
        "uT_test_fine": uT_fine[n_train:],
    }


def timed(fn, dev: torch.device, repeats: int = 3) -> float:
    """Best wall-clock time of fn() in seconds."""
    best = float("inf")
    for _ in range(repeats):
        if dev.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            fn()
        if dev.type == "cuda":
            torch.cuda.synchronize()
        best = min(best, time.perf_counter() - t0)
    return best


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    nu, t_final, n_train = params["nu"], params["t_final"], params["n_train"]

    t0 = time.perf_counter()
    data = make_data(n_train, nu, t_final, dev)
    t_data = time.perf_counter() - t0
    # Super-resolution panel: the test case with the steepest front at time T, at full resolution.
    slope = data["uT_test_fine"].diff(dim=1).abs().amax(dim=1) / data["dx_fine"]  # max |u_x| per case
    data["fine_idx"] = int(slope.argmax())
    data["x_fine32"] = data["x_fine"].float()
    data["u0_fine"] = data["u0_test_fine"][[data["fine_idx"]]].float()  # (1, M)

    w = tensors.init_fno(params["width"], params["modes"], N_LAYERS, SEED, dev)
    t0 = time.perf_counter()
    hist = solve.train(w, data, params["epochs"], params["lr"], BATCH, SEED)
    t_train = time.perf_counter() - t0

    # Identity baseline "u(T) = u_0", and the zero-shot super-resolution error on the 256-point grid.
    u0_fine32, uT_fine32 = data["u0_test_fine"].float(), data["uT_test_fine"].float()
    with torch.no_grad():
        baseline = equation.relative_l2(data["u0_test"], data["uT_test"]).item()
        fine_err = equation.relative_l2(equation.fno(u0_fine32, data["x_fine32"], w), uT_fine32).item()

    # Timing: the solver vs one FNO forward pass, for all N_TEST test initial conditions at once.
    t_solver = timed(lambda: solve.solve_burgers(data["u0_test_fine"], nu, t_final, data["dx_fine"]), dev)
    t_fno = timed(lambda: equation.fno(data["u0_test"], data["x"], w), dev)
    t_fno_fine = timed(lambda: equation.fno(u0_fine32, data["x_fine32"], w), dev)

    print(f"Burgers u_t + u u_x = ν u_xx, ν = {nu}, T = {t_final}   ·   {n_train} training / {N_TEST} test "
          f"initial conditions, solved on {N_FINE} points in {t_data:.1f} s")  # fmt: skip
    print(f"FNO: width {params['width']}, {params['modes']} Fourier modes, {N_LAYERS} Fourier layers, "
          f"{tensors.count(w):,} real parameters, trained {params['epochs']} epochs in {t_train:.1f} s")  # fmt: skip
    print(f"relative L2 error ({N_COARSE} pts)   train {hist['train_err'][-1]:.2%}   test {hist['test_err'][-1]:.2%}"
          f"   (baseline 'u(T) = u_0': {baseline:.2%})")  # fmt: skip
    print(f"zero-shot super-resolution: the same weights on {N_FINE} points, test error {fine_err:.2%}")
    print(f"time for all {N_TEST} test cases:  solver ({N_FINE} pts) {t_solver * 1e3:.1f} ms   ·   "
          f"FNO ({N_COARSE} pts) {t_fno * 1e3:.1f} ms   ·   FNO ({N_FINE} pts) {t_fno_fine * 1e3:.1f} ms")  # fmt: skip

    return plot.animate(data, hist, baseline, fine_err, out_dir / "neural_operator.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/others/neural_operator/equation.py
r"""Physics → tensor translation.

The physics is the viscous Burgers equation on a periodic domain x ∈ [0, 2π),

    u_t + u u_x = ν u_xx      ⇔      û_t(k) = -ν k² û(k) - (ik/2) FFT[u²](k)

written in Fourier space (k = wavenumber), where ∂_x becomes multiplication by ik.

A neural OPERATOR learns the map G: u_0(·) ↦ u(·, T) for a whole family of initial conditions. A Fourier
neural operator (FNO) acts on a function sampled at N grid points x, with indices
b = sample in the batch, i/o = input/output channel, x = grid point, k = Fourier mode:

    lift           v_{bcx}   = P_{cf} a_{bfx} + p_c                      f summed   (a = [u_0, x/2π])
    Fourier layer  v'_{box}  = σ( W_{oi} v_{bix} + b_o + (K v)_{box} )   i summed
    spectral conv  (K v)     = IFFT_k[ R_{iok} v̂_{bik} ]  for k < m,     i summed, modes k ≥ m dropped
    project        u_{bx}    = Q2_{h} σ( Q1_{hc} v_{bcx} + q1_h ) + q2   c, h summed

The weights R_{iok} are complex numbers that live on Fourier MODES, not on grid points: the same
weights work on any grid with more than 2m points. That is why the FNO is discretisation invariant.
"""

import torch
import torch.nn.functional as F

# --- the PDE (used by the data solver) ---------------------------------------------------------------


def burgers_nonlinear(u_hat: torch.Tensor, k: torch.Tensor, dealias: torch.Tensor, n: int) -> torch.Tensor:
    r"""-(ik/2) FFT[u²]: the advection term u u_x = (½u²)_x in Fourier space.   u_hat: (B, K) → (B, K).

    u² is formed on the grid (pseudo-spectral); modes |k| > N/3 are zeroed (2/3 rule) against aliasing.
    """
    u = torch.fft.irfft(u_hat, n=n)  # (B, N) back to grid values
    return -0.5j * k * dealias * torch.fft.rfft(u**2)


def mass(u: torch.Tensor, dx: float) -> torch.Tensor:
    r"""∫ u dx ≈ Σ_x u_x Δx — conserved: the k = 0 mode is untouched by ik(·) and by νk²."""
    return u.sum(dim=-1) * dx


# --- the Fourier neural operator ---------------------------------------------------------------------


def lift(a: torch.Tensor, P: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    """v_{bcx} = P_{cf} a_{bfx} + p_c   (f summed): 2 input features → C channels at every point x."""
    return torch.einsum("cf,bfx->bcx", P, a) + p[:, None]


def spectral_conv(v: torch.Tensor, R: torch.Tensor) -> torch.Tensor:
    r"""(K v)_{box} = IFFT[ R_{iok} v̂_{bik} ]   (i summed), keeping only the lowest m = R.shape[-1] modes.

    v: (B, C, N) real,  R: (C, C, m) complex  →  (B, C, N) real.
    """
    n, m = v.shape[-1], R.shape[-1]
    v_hat = torch.fft.rfft(v)  # (B, C, N//2 + 1) complex Fourier coefficients
    out_hat = torch.zeros_like(v_hat)  # modes k ≥ m stay zero: the truncation
    out_hat[..., :m] = torch.einsum("bik,iok->bok", v_hat[..., :m], R)  # mix channels, mode by mode
    return torch.fft.irfft(out_hat, n=n)  # back to the same N grid points


def fourier_layer(v: torch.Tensor, R: torch.Tensor, W: torch.Tensor, b: torch.Tensor, last: bool) -> torch.Tensor:
    """v' = σ(W v + b + K v): a local (pointwise) path plus a global (spectral) path.   (B, C, N) → (B, C, N)."""
    local = torch.einsum("oi,bix->box", W, v) + b[:, None]  # i summed, same W at every x
    out = local + spectral_conv(v, R)
    return out if last else F.gelu(out)  # σ = GELU (no activation after the last layer)


def project(v: torch.Tensor, Q1, q1, Q2, q2) -> torch.Tensor:
    """u_{bx} = Q2_h σ(Q1_{hc} v_{bcx} + q1_h) + q2   (c, h summed): C channels → 1 value per point."""
    h = F.gelu(torch.einsum("hc,bcx->bhx", Q1, v) + q1[:, None])
    return torch.einsum("h,bhx->bx", Q2, h) + q2


def fno(u0: torch.Tensor, x: torch.Tensor, w: dict) -> torch.Tensor:
    """The operator G_w: u_0 sampled on ANY periodic grid x → u(·, T) on the same grid.   u0: (B, N), x: (N,)."""
    grid = (x / (2 * torch.pi)).expand_as(u0)  # (B, N) the position as a second input feature
    a = torch.stack([u0, grid], dim=1)  # (B, 2, N)
    v = lift(a, w["P"], w["p"])  # (B, C, N)
    n_layers = len(w["R"])
    for layer in range(n_layers):
        v = fourier_layer(v, w["R"][layer], w["W"][layer], w["b"][layer], last=layer == n_layers - 1)
    return project(v, w["Q1"], w["q1"], w["Q2"], w["q2"])  # (B, N)


def relative_l2(pred: torch.Tensor, true: torch.Tensor) -> torch.Tensor:
    """‖pred - true‖ / ‖true‖ per sample, averaged over the batch: the training loss.   (B, N) → ()."""
    return (torch.linalg.norm(pred - true, dim=-1) / torch.linalg.norm(true, dim=-1)).mean()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/others/neural_operator/tensors.py
"""Build the concrete tensors: periodic grids, random initial conditions and the FNO weights."""

import math

import torch

LENGTH = 2 * math.pi  # periodic domain [0, 2π)


def grid(n: int, device: torch.device) -> tuple[torch.Tensor, float]:
    """N equally spaced points x on [0, 2π) (float64) and the spacing Δx."""
    dx = LENGTH / n
    return torch.arange(n, device=device, dtype=torch.float64) * dx, dx


def random_modes(n_samples: int, n_modes: int, seed: int, device: torch.device) -> torch.Tensor:
    """Random Fourier coefficients c_{bk} = (a_bk, b_bk) for k = 1 … n_modes: shape (B, n_modes, 2).

    Amplitudes fall off like 1/k so the initial conditions are smooth; the mean (k = 0) is zero.
    """
    gen = torch.Generator().manual_seed(seed)
    k = torch.arange(1, n_modes + 1, dtype=torch.float64)
    coeffs = torch.randn(n_samples, n_modes, 2, generator=gen, dtype=torch.float64) * 0.6 / k[:, None]
    return coeffs.to(device)


def initial_conditions(coeffs: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """u_0(x) = Σ_k a_k cos kx + b_k sin kx, sampled on the grid x: (B, K, 2), (N,) → (B, N).

    The same coefficients evaluated on a finer grid give the SAME functions at higher resolution.
    """
    k = torch.arange(1, coeffs.shape[1] + 1, device=x.device, dtype=x.dtype)
    kx = k[:, None] * x[None, :]  # (K, N)
    return torch.einsum("bk,kn->bn", coeffs[..., 0], torch.cos(kx)) + torch.einsum(
        "bk,kn->bn", coeffs[..., 1], torch.sin(kx)
    )


def init_fno(width: int, n_modes: int, n_layers: int, seed: int, device: torch.device) -> dict:
    """FNO weights (all trainable):

    P (C, 2), p (C,)                        lift: [u_0, x] → C channels
    R[l] (C, C, m) complex, W[l] (C, C), b[l] (C,)   Fourier layer l = 0 … L-1
    Q1 (H, C), q1 (H,), Q2 (H,), q2 ()      projection: C → H = 2C → 1
    """
    gen = torch.Generator().manual_seed(seed)

    def dense(*shape):  # uniform ±1/√fan_in, like torch.nn.Linear
        fan_in = shape[-1] if len(shape) > 1 else width
        return (torch.rand(*shape, generator=gen) * 2 - 1) / math.sqrt(fan_in)

    def spectral():  # small random complex weights, as in the original FNO paper
        scale = 1.0 / (width * width)
        return scale * torch.complex(
            torch.rand(width, width, n_modes, generator=gen), torch.rand(width, width, n_modes, generator=gen)
        )

    hidden = 2 * width
    w = {
        "P": dense(width, 2),
        "p": dense(width, 2)[:, 0],
        "R": [spectral() for _ in range(n_layers)],
        "W": [dense(width, width) for _ in range(n_layers)],
        "b": [dense(width, width)[:, 0] for _ in range(n_layers)],
        "Q1": dense(hidden, width),
        "q1": dense(hidden, width)[:, 0],
        "Q2": dense(1, hidden)[0],
        "q2": torch.zeros(()),
    }
    return {
        key: [t.to(device).requires_grad_(True) for t in val]
        if isinstance(val, list)
        else val.to(device).requires_grad_(True)
        for key, val in w.items()
    }


def parameters(w: dict) -> list[torch.Tensor]:
    """Flat list of every trainable tensor (for the optimiser)."""
    return [t for val in w.values() for t in (val if isinstance(val, list) else [val])]


def count(w: dict) -> int:
    """Number of real trainable numbers (a complex weight counts twice)."""
    return sum(t.numel() * (2 if t.is_complex() else 1) for t in parameters(w))


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/others/neural_operator/solve.py
"""The data solver (pseudo-spectral Burgers, a whole batch at once) and the FNO training loop (Adam)."""

import math

import torch

from problems._common.progress import steps

from .equation import burgers_nonlinear, fno, relative_l2
from .tensors import parameters


def solve_burgers(u0: torch.Tensor, nu: float, t_final: float, dx: float) -> torch.Tensor:
    r"""u(·, T) for every initial condition in the batch: u0 (B, N) → (B, N).

    Integrating-factor RK4 in Fourier space: the stiff diffusion -νk² û is solved exactly by the factor
    e^{-νk²Δt}, RK4 handles only the advection, so Δt is limited by Δx / max|u| alone.
    """
    n = u0.shape[-1]
    k = torch.fft.rfftfreq(n, d=dx / (2 * math.pi)).to(u0.device)  # wavenumbers 0, 1, …, N/2   (K,)
    dealias = (k < n / 3).to(u0.dtype)  # 2/3 rule
    u_max = u0.abs().max().item() + 1e-12
    n_steps = max(1, math.ceil(t_final / (0.5 * dx / u_max)))
    dt = t_final / n_steps
    E, E2 = torch.exp(-nu * k**2 * dt), torch.exp(-nu * k**2 * dt / 2)  # integrating factors (K,)

    def N(v):
        return burgers_nonlinear(v, k, dealias, n)

    u_hat = torch.fft.rfft(u0)  # (B, K)
    for _ in steps(n_steps, "solver steps"):
        a = dt * N(u_hat)
        b = dt * N(E2 * (u_hat + a / 2))
        c = dt * N(E2 * u_hat + b / 2)
        d = dt * N(E * u_hat + E2 * c)
        u_hat = E * u_hat + (E * a + 2 * E2 * (b + c) + d) / 6
    return torch.fft.irfft(u_hat, n=n)


def train(w: dict, data: dict, epochs: int, lr: float, batch: int, seed: int, label: str = "epochs") -> dict:
    """Adam on the relative L2 loss, in mini-batches of shuffled training samples. data holds float32 tensors
    x (N,), u0_train/uT_train (n_train, N), u0_test/uT_test (n_test, N) and, optionally, for the
    super-resolution panel, u0_fine (S, M) on the finer grid x_fine32 (M,).

    Returns the train/test error after every epoch (entry 0 = untrained) and, for the animation, the
    predictions on the whole test set (coarse grid) and on u0_fine (fine grid) after every epoch.
    """
    gen = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam(parameters(w), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs, eta_min=lr / 20)
    x, u0, uT = data["x"], data["u0_train"], data["uT_train"]
    fine = "u0_fine" in data

    def evaluate(train_err: float) -> dict:
        with torch.no_grad():
            pred_test = fno(data["u0_test"], x, w)
            return {
                "train_err": train_err,
                "test_err": relative_l2(pred_test, data["uT_test"]).item(),
                "pred_test": pred_test.cpu(),  # (n_test, N)
                "pred_fine": fno(data["u0_fine"], data["x_fine32"], w).cpu() if fine else torch.zeros(0),
            }

    with torch.no_grad():
        history = [evaluate(relative_l2(fno(u0, x, w), uT).item())]  # entry 0: the untrained model
    for _ in steps(epochs, label):
        order = torch.randperm(len(u0), generator=gen).to(u0.device)  # shuffle the training set
        losses = []
        for start in range(0, len(u0), batch):
            idx = order[start : start + batch]
            loss = relative_l2(fno(u0[idx], x, w), uT[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        sched.step()
        history.append(evaluate(sum(losses) / len(losses)))  # train error = mean batch loss of the epoch

    stack = {"train_err": torch.tensor, "test_err": torch.tensor, "pred_test": torch.stack, "pred_fine": torch.stack}
    return {key: fn([h[key] for h in history]) for key, fn in stack.items()}  # each with leading dim epochs + 1


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/others/neural_operator/plot.py
"""Animate training: FNO predictions on unseen initial conditions, the error curves, and zero-shot super-resolution."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import frame_indices, save_gif

BLUE, RED, ORANGE, GREY, DARK = "#2563eb", "#dc2626", "#ea580c", "#9ca3af", "#374151"


def animate(data: dict, hist: dict, baseline: float, fine_err: float, path: Path) -> Path:
    x = data["x"].cpu().numpy()  # (N,)
    xf = data["x_fine"].cpu().numpy()  # (M,)
    show = [0, 1, 2]  # the first three test cases
    u0 = data["u0_test"][show].cpu().numpy()  # (3, N)
    uT = data["uT_test"][show].cpu().numpy()
    k = data["fine_idx"]  # the test case with the steepest front
    uT_fine = data["uT_test_fine"][k].cpu().numpy()  # (M,) at full resolution
    pred_coarse = hist["pred_test"][:, k].numpy()  # (E+1, N) the same case on the training grid
    pred = hist["pred_test"][:, show].numpy()  # (E+1, 3, N)
    pred_fine = hist["pred_fine"][:, 0].numpy()  # (E+1, M)
    train_err, test_err = hist["train_err"].numpy(), hist["test_err"].numpy()
    epochs = np.arange(len(train_err))
    lim = 1.15 * max(np.abs(u0).max(), np.abs(uT).max())

    fig = plt.figure(figsize=(8.0, 4.6))
    grid = fig.add_gridspec(2, 3, height_ratios=[1, 1.1])
    tops = [fig.add_subplot(grid[0, j]) for j in range(3)]
    ax_e = fig.add_subplot(grid[1, 0])
    ax_s = fig.add_subplot(grid[1, 1:])

    # --- three unseen test initial conditions: u0 (dashed), true u(T) (grey), FNO (blue) ---
    lines = []
    for j, ax in enumerate(tops):
        ax.plot(x, u0[j], "--", color=GREY, lw=1.2, label="$u_0$")
        ax.plot(x, uT[j], color=DARK, lw=3, alpha=0.35, label="true $u(T)$")
        lines.append(ax.plot([], [], color=BLUE, lw=1.6, label="FNO")[0])
        ax.set_xlim(0, 2 * np.pi)
        ax.set_ylim(-lim, lim)
        ax.set_xticks([0, np.pi, 2 * np.pi], ["0", "π", "2π"])
        ax.tick_params(labelsize=8)
        ax.set_title(f"unseen test case {j + 1}", fontsize=9)
    tops[0].legend(loc="lower right", fontsize=6.5)

    # --- relative L2 error vs epoch ---
    ax_e.set_yscale("log")
    ax_e.minorticks_off()
    ax_e.set_xlim(0, epochs[-1])
    lo = min(train_err.min(), test_err.min())
    ax_e.set_ylim(lo * 0.6, max(train_err.max(), test_err.max(), baseline) * 1.6)
    ax_e.axhline(baseline, color=GREY, ls="--", lw=1)
    ax_e.text(epochs[-1] * 0.97, baseline * 1.1, "$u(T) = u_0$", ha="right", va="bottom", fontsize=7, color="#6b7280")
    (train_line,) = ax_e.plot([], [], color=BLUE, lw=1.4, label="train")
    (test_line,) = ax_e.plot([], [], color=RED, lw=1.4, label="test")
    ax_e.set_xlabel("epoch", fontsize=8)
    ax_e.set_title("relative $L^2$ error", fontsize=9)
    ax_e.tick_params(labelsize=7)
    ax_e.legend(loc="center right", fontsize=7)

    # --- zero-shot super-resolution: same weights on 256 points ---
    ax_s.plot(xf, uT_fine, color=DARK, lw=3, alpha=0.35, label=f"solver, {len(xf)} pts")
    (fine_line,) = ax_s.plot([], [], color=BLUE, lw=1.4, label=f"FNO on {len(xf)} pts (zero-shot)")
    (coarse_pts,) = ax_s.plot([], [], "o", color=ORANGE, ms=3, label=f"FNO on {len(x)} pts (training grid)")
    ax_s.set_xlim(0, 2 * np.pi)
    lim_s = 1.15 * np.abs(uT_fine).max()
    ax_s.set_ylim(-lim_s, lim_s)
    ax_s.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi], ["0", "π/2", "π", "3π/2", "2π"])
    ax_s.tick_params(labelsize=7)
    ax_s.set_title(f"super-resolution: steepest test case, trained on {len(x)} points, run on {len(xf)}", fontsize=9)
    ax_s.legend(loc="upper left", fontsize=6.5)
    case_err = np.linalg.norm(pred_fine[-1] - uT_fine) / np.linalg.norm(uT_fine)
    counter = fig.text(0.5, 0.005, "", ha="center", va="bottom", fontsize=9)

    def update(e: int):
        for j, line in enumerate(lines):
            line.set_data(x, pred[e, j])
        train_line.set_data(epochs[: e + 1], train_err[: e + 1])
        test_line.set_data(epochs[: e + 1], test_err[: e + 1])
        fine_line.set_data(xf, pred_fine[e])
        coarse_pts.set_data(x, pred_coarse[e])
        text = f"epoch {e}   ·   test error {test_err[e]:.1%}"
        if e == len(train_err) - 1:  # the super-resolution errors refer to the final model
            n_test = len(data["u0_test"])
            text += f"   ·   on {len(xf)} pts: all {n_test} test cases {fine_err:.1%}, steepest {case_err:.1%}"
        counter.set_text(text)

    fig.tight_layout(rect=(0, 0.04, 1, 1))
    return save_gif(fig, update, frame_indices(len(train_err), 30), path, fps=6)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'nu': 0.05,
    't_final': 1.0,
    'n_train': 800,
    'modes': 12,
    'width': 24,
    'epochs': 40,
    'lr': 0.003
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.others.neural_operator")]:
    del sys.modules[name]
main = importlib.import_module("problems.others.neural_operator.main")
problem_dir = Path("problems/others/neural_operator")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
